NOTE: From Exploratory Data Analysis:

Dataset has 11 (1.1%) duplicate rows i.e DUPLICATE GROUPS.

age is highly overall correlated with concrete_compressive_strength.

concrete_compressive_strength is highly overall correlated with age.

superplasticizer is highly overall correlated with water.

water is highly overall correlated with superplasticizer.

blast_furnace_slag has 471 (45.7%) zeros.
fly_ash has 566 (55.0%) zeros.

superplasticizer has 379 (36.8%) zeros.

In [ ]:
# dependencies
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

In [ ]:
# reproducibility


In [ ]:
# Re-establish reproducibility seeds and configure folder paths
seed = 2006
np.random.seed(seed)

# Configure local folder structure for this workspace
base_path = Path('/content/drive/MyDrive/Concrete Compressive Strength Prediction')
raw_data_dir = base_path / '01_Dataset' / 'Raw'
processed_data_dir = base_path / '01_Dataset' / 'Processed'
fig_pre_dir = base_path / '03_Figures' / 'pre-processing'

# Ensure the directories exist
os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(processed_data_dir, exist_ok=True)
os.makedirs(fig_pre_dir, exist_ok=True)

# Download the dataset directly from the UCI Machine Learning Repository
# The dataset is stored as an Excel (.xls) file on UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
raw_data_path = raw_data_dir / 'concrete_data.csv'

try:
    # Read the excel file directly from the URL and save it as a CSV
    df_raw = pd.read_excel(url)
    df_raw.to_csv(raw_data_path, index=False)
    print(f"Dataset downloaded successfully and saved to: {raw_data_path}")
    display(df_raw.head())
    print("Shape:", df_raw.shape)
except Exception as e:
    print(f"An error occurred while downloading: {e}")

In [ ]:
df_raw.describe()

In [ ]:
df_raw.info() # no nullity as already observed in EDA

In [ ]:
df_raw.duplicated().value_counts()

In [ ]:
df_clean = df_raw.copy()
df_clean.drop_duplicates(inplace=True)

In [ ]:
df_clean.duplicated().value_counts()

In [ ]:
# Outlier detection
# visualization
plt.figure(figsize=(20, 15))

# There are 9 columns, so we can use a 3x3 subplot grid
for i, column in enumerate(df_clean.columns):
    plt.subplot(3, 3, i + 1)
    sns.boxplot(y=df_clean[column])
    plt.title(f'Box plot of {column}')
    plt.ylabel('Value')

plt.tight_layout()
plt.savefig(fig_pre_dir / 'boxplot.png')
plt.show()

In [ ]:
# Find the actual column names containing 'water' and 'cement'
water_col = [col for col in df_clean.columns if 'water' in col.lower()][0]
cement_col = [col for col in df_clean.columns if 'cement' in col.lower()][0]

# Safely calculate the water-to-cement ratio
df_clean['water_cement_ratio'] = df_clean[water_col] / df_clean[cement_col].replace(0, np.nan)
print("water_cement_ratio created successfully!")
print(df_clean[['water_cement_ratio']].head())

In [ ]:
df_clean.sample(5)

Comprehensive Data Visualizations

To understand the data distribution, relationships between variables, and potential patterns, we will generate the following visualizations:

Histograms and KDE plots: To visualize the distribution of each feature.

Correlation Heatmap: To show the linear relationships between all numerical features.

Scatter Plots: To examine the relationship between individual features and the target variable (concrete_compressive_strength).

1. Distribution of Features (Histograms and KDE plots)

In [ ]:
plt.figure(figsize=(20, 15))
for i, column in enumerate(df_clean.columns):
    plt.subplot(3, 4, i + 1) # Adjust subplot grid to fit the new water_cement_ratio column
    sns.histplot(df_clean[column], kde=True)
    plt.title(f'Distribution of {column}')
    plt.xlabel(column)
    plt.ylabel('Frequency')
plt.tight_layout()
output_filepath = fig_pre_dir / 'feature_distributions.png'
output_filepath.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_filepath)
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df_clean.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Features')
plt.tight_layout()
output_filepath = fig_pre_dir / 'correlation_heatmap.png'
plt.savefig(output_filepath)
plt.show()

In [ ]:
# Dynamically find the target column containing 'strength'
target_variable = [col for col in df_clean.columns if 'strength' in col.lower()][0]
features = [col for col in df_clean.columns if col != target_variable]

plt.figure(figsize=(20, 20))
for i, feature in enumerate(features):
    plt.subplot(4, 3, i + 1)
    sns.scatterplot(x=df_clean[feature], y=df_clean[target_variable])
    plt.title(f'{feature} vs Target')
    plt.xlabel(feature)
    plt.ylabel('Concrete Compressive Strength')
plt.tight_layout()
output_filepath = fig_pre_dir / 'feature_vs_target_scatterplots.png'
plt.savefig(output_filepath)
plt.show()

In [ ]:
# save the df_clean csv
df_clean.to_csv(processed_data_dir / 'df_clean.csv', index=False)

In [ ]:
from google.colab import files

# Path of the saved processed CSV
csv_path = processed_data_dir / 'df_clean.csv'

# Download the file to your local system
if csv_path.exists():
    files.download(str(csv_path))
else:
    print(f"File not found at: {csv_path}")

In [ ]:
import shutil
from google.colab import files

# Create a zip of all processed results (Processed Data + Figures)
export_dir = Path('/content/processed_results')
export_dir.mkdir(exist_ok=True)

# Copy CSV
shutil.copy(processed_data_dir / 'df_clean.csv', export_dir / 'df_clean.csv')

# Copy Figures
fig_export_dir = export_dir / 'figures'
if fig_pre_dir.exists():
    shutil.copytree(fig_pre_dir, fig_export_dir, dirs_exist_ok=True)

# Zip and download
zip_path = shutil.make_archive('/content/all_processed_data', 'zip', export_dir)
files.download(zip_path)